<a href="https://colab.research.google.com/github/duyilemi/mistral_alpaca_qlora/blob/main/mistral_alpaca_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installations and Authentication

In [ ]:
!pip install -q \
    transformers \
    accelerate \
    "datasets>=2.18.0" \
    fsspec==2023.9.2 \
    gcsfs==2023.9.2 \
    peft \
    bitsandbytes \
    trl \
    huggingface_hub \
    tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.4/173.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Authenticate with Hugging Face Hub
import torch
%env HF_HOME=/root/.cache/huggingface
from huggingface_hub import notebook_login
notebook_login()

torch_device = "cuda" if torch.cuda.is_available() else "cpu"

env: HF_HOME=/root/.cache/huggingface


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Imports & Configuration

In [ ]:
import math
import gc
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

MODEL_ID = "mistralai/Mistral-7B-v0.1"
DATASET_ID = "yahma/alpaca-cleaned"
OUTPUT_DIR = "/content/drive/MyDrive/mistral-alpaca-qlora"

# Data Preparation

In [ ]:
# Load a small subset and split into train/test
dataset = load_dataset(DATASET_ID, split="train[:5%]").train_test_split(test_size=0.1, seed=42)
print(dataset)
print("Columns:", dataset['train'].column_names)

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 2329
    })
    test: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 259
    })
})
Columns: ['output', 'input', 'instruction']


In [ ]:
# View first 5 examples
print("Sample of 5 raw entries:", dataset['train'].select(range(5)))

Sample of 5 raw entries: Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 5
})


In [ ]:
# Formatting function setup
def format_alpaca(sample):
    return (
        "[INST] <<SYS>>\n"
        "You are a helpful AI assistant. Follow instructions carefully.\n"
        "<</SYS>>\n\n"
        f"{sample['instruction']}\n{sample['input']} [/INST] {sample['output']}</s>"
    )

In [ ]:
# Sample dataset
sample = dataset['train'][0]
print("Original sample:", sample)

Original sample: {'output': 'The lost key was found by her.', 'input': 'She found the lost key.', 'instruction': 'Rewrite the sentence in the passive voice.'}


In [ ]:

# Test formatting function
formatted = format_alpaca(sample)
print("Formatted text sample:", formatted)

Formatted text sample: [INST] <<SYS>>
You are a helpful AI assistant. Follow instructions carefully.
<</SYS>>

Rewrite the sentence in the passive voice.
She found the lost key. [/INST] The lost key was found by her.</s>


In [ ]:
# Setup tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
# Test tokenizer
tokenized = tokenizer(
    formatted,
    padding="max_length",
    truncation=True,
    max_length=512,
)
print("Tokenized input ids:", tokenized['input_ids'][:10], "... length:", len(tokenized['input_ids']))

Tokenized input ids: [1, 733, 16289, 28793, 2087, 18741, 4060, 13, 1976, 460] ... length: 512


# Model Setup

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# LoRA & SFT configuration

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
sft_config = SFTConfig(
    max_seq_length=512,
    dataset_text_field="text",
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    num_train_epochs=1,
    max_steps=300,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="paged_adamw_8bit",
    fp16=True,
    gradient_checkpointing=True,
    max_grad_norm=0.5,
    warmup_ratio=0.1,
    report_to="tensorboard"
)

# Validation Checks

In [ ]:
def validate_pipeline():
    sample = dataset['train'][0]
    formatted = format_alpaca(sample)
    assert isinstance(formatted, str)
    tokenized = tokenizer(
        formatted,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    assert tokenized['input_ids'].shape == torch.Size([1, 512])
    collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
    batch = collator([{
        'input_ids': tokenized['input_ids'][0],
        'attention_mask': tokenized['attention_mask'][0]
    }])
    assert batch['input_ids'].shape == torch.Size([1, 512])
    print("Validation checks passed!")

validate_pipeline()

Validation checks passed!


# Trainer Initialization

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=peft_config,
    formatting_func=format_alpaca,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        pad_to_multiple_of=8
    )
)

Applying formatting function to train dataset:   0%|          | 0/2329 [00:00<?, ? examples/s]

Converting train dataset to ChatML:   0%|          | 0/2329 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/2329 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2329 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2329 [00:00<?, ? examples/s]

...

Converting eval dataset to ChatML:   0%|          | 0/259 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/259 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/259 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/259 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


# Tensorboard Setup

In [ ]:
%load_ext tensorboard
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
%tensorboard --logdir "$OUTPUT_DIR/runs"

# Training and Evaluation

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print("\nStarting training...")
trainer.train()


Starting training...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
50,0.924500,0.969819
100,0.903200,0.957055
150,0.888200,0.944791
200,0.837600,0.924618
250,0.844600,0.909650
300,0.841000,0.907344


TrainOutput(global_step=300, training_loss=0.9240155919392904, metrics={'train_runtime': 2246.4616, 'train_samples_per_second': 0.534, 'train_steps_per_second': 0.134, 'total_flos': 1.4078342120865792e+16, 'train_loss': 0.9240155919392904})

In [ ]:
metrics = trainer.evaluate()

In [ ]:
print(metrics)

{'eval_loss': 0.9073435664176941, 'eval_runtime': 106.09, 'eval_samples_per_second': 2.441, 'eval_steps_per_second': 0.311}


In [ ]:
perplexity = math.exp(metrics['eval_loss'])
print(f"Evaluation Loss: {metrics['eval_loss']:.4f}, Perplexity: {perplexity:.2f}")

Evaluation Loss: 0.9073, Perplexity: 2.48


# Qualitative Tests

In [ ]:
prompts = [
    "Explain quantum entanglement in simple terms.",
    "How to make a perfect omelette?"
]
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    base_output = model.generate(**inputs, max_new_tokens=50)
    print(f"\nPrompt: {prompt}")
    print("Generated:", tokenizer.decode(base_output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Prompt: Explain quantum entanglement in simple terms.
Generated: Explain quantum entanglement in simple terms.

Quantum entanglement is a phenomenon in which two or more particles are linked in such a way that the quantum state of each particle cannot be described independently of the others, even when the particles are separated by a large distance. This means

Prompt: How to make a perfect omelette?
Generated: How to make a perfect omelette? [INST] [/INST]To make a perfect omelette, follow these steps: 1. Crack the eggs into a bowl and whisk them until they are well combined. 2. Heat a non-stick pan over medium heat


# Save and push to hub

In [ ]:
tokenizer.save_pretrained(OUTPUT_DIR)
trainer.model.save_pretrained(OUTPUT_DIR)

In [ ]:

REPO_ID = "duyilemi/mistral-alpaca-qlora"

trainer.model.push_to_hub(REPO_ID,
    use_temp_dir=True,
    commit_message="Add trained LoRA adapter",     safe_serialization=True)
tokenizer.push_to_hub(REPO_ID,
    use_temp_dir=True,
    commit_message="Add tokenizer config")

adapter_model.safetensors:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/duyilemi/mistral-alpaca-qlora/commit/87bd12ead5c6a774e547b330dd0a2d459924ae2e', commit_message='Add tokenizer config', commit_description='', oid='87bd12ead5c6a774e547b330dd0a2d459924ae2e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyilemi/mistral-alpaca-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='duyilemi/mistral-alpaca-qlora'), pr_revision=None, pr_num=None)